In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm

# Add parent directory to sys.path if needed
parent_dir = os.path.join(os.getcwd(), '..')
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

from utility.data import get_loader, EmbeddingDataset
from utility.visuals import *
from mkit.torch_support.tensor_utils import xy_to_tensordataset
from model.gan import GANs
from model.knn import KNN
from model.rf import RF
from model.dnn import DNN
from model.data_augmentation_tech import (
    conditional_smote_sampling, 
    conditional_kde_sampling, 
    conditional_gmm_sampling
)
import torch
from torch.utils.data import Subset
from datasets import load_from_disk

# Device and parameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
latent_dim = 128          
condition_dim = 10        
gan_epochs = 200  
generation_sizes = [1000, 10000]  # Multiple generation sizes for synthetic generation
# generation_sizes = [100]
batch_size = 32                
num_epochs = 10
learning_rate = 0.001

# Load dataset and split into train and test
full_dataset = load_from_disk('../data/full_dataset_new', keep_in_memory=True)
split_datasets = full_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_datasets['train']
test_dataset = split_datasets['test']

# For external test evaluation (if desired)
test_x = np.array(test_dataset['embedding'])
test_text = np.array(test_dataset['input_ids'])
test_y = np.array(test_dataset['labels'])

# Full training arrays (will be used for selecting sample sizes)
train_x_full = np.array(train_dataset['embedding'])
train_text = np.array(train_dataset['input_ids'])
train_y_full = np.array(train_dataset['labels'])
train_ds = EmbeddingDataset(train_dataset)
test_ds = EmbeddingDataset(test_dataset)

# Get input dimension from one sample
sample_emb, _ = train_ds[0]
input_dim = sample_emb.shape[0]
hidden_dim = 128
num_classes = 3  

In [ ]:
from transformers import AutoTokenizer
model_name = "tasksource/deberta-small-long-nli"
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')


In [ ]:
from utility.eval import *

In [ ]:

from sklearn.multioutput import MultiOutputClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
pred = MultiOutputClassifier(KNeighborsClassifier(n_neighbors=1))
pred.fit([[1, 2, 3,]], [[4, 5, 6]])
pred.predict([[1, 2, 3]])

In [ ]:
# import numpy as np
# from scipy.spatial.distance import cdist
# from scipy.optimize import linear_sum_assignment
# from IPython.display import clear_output
# def align_sequences(seq1, seq2, metric='euclidean', one_to_one=True):
#     """
#     Aligns two sequences of vectors based on their closest counterparts, preserving original format.

#     Parameters:
#     - seq1: np.ndarray or torch.Tensor, shape (n, d), first sequence of vectors
#     - seq2: np.ndarray or torch.Tensor, shape (m, d), second sequence of vectors
#     - metric: str, distance metric (default: 'euclidean')
#     - one_to_one: bool, whether to perform one-to-one alignment using Hungarian algorithm (default: True)

#     Returns:
#     - aligned_seq1: np.ndarray or torch.Tensor, aligned vectors from seq1
#     - aligned_seq2: np.ndarray or torch.Tensor, corresponding closest vectors from seq2
#     """
#     # Detect input types and convert to numpy if necessary
#     seq1_is_tensor = isinstance(seq1, torch.Tensor)
#     seq2_is_tensor = isinstance(seq2, torch.Tensor)

#     if seq1_is_tensor:
#         seq1 = seq1.cpu().numpy()
#     if seq2_is_tensor:
#         seq2 = seq2.cpu().numpy()

#     # Compute pairwise distance matrix
#     cost_matrix = cdist(seq1, seq2, metric=metric)

#     if one_to_one:
#         # Use Hungarian algorithm for optimal one-to-one alignment
#         row_ind, col_ind = linear_sum_assignment(cost_matrix)
#         aligned_seq1 = seq1[row_ind]
#         aligned_seq2 = seq2[col_ind]
#     else:
#         # Use Nearest Neighbors (one-to-many alignment)
#         nearest_indices = np.argmin(cost_matrix, axis=1)
#         aligned_seq1 = seq1
#         aligned_seq2 = seq2[nearest_indices]

#     # Convert back to tensors if original inputs were tensors
#     if seq1_is_tensor:
#         aligned_seq1 = torch.tensor(aligned_seq1)
#     if seq2_is_tensor:
#         aligned_seq2 = torch.tensor(aligned_seq2)

#     return aligned_seq1, aligned_seq2



In [ ]:

model = SentenceTransformer('all-MiniLM-L6-v2')

# Number of folds for cross validation
n_splits = 5

# List to collect results; each entry will be a dict for one (sample_size, generation_size) combo.
summary_data = []
generation_size = 10
size = 80
print(f"\n[Train Data] Sample size: {size}")
# Get the first "size" samples from the full training set
X_train = train_x_full[:size]
y_train = train_y_full[:size]
y_train_text = train_text[:size]
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)


fold_no = 1
for train_index, val_index in skf.split(X_train, y_train):
    print(f"Fold: [{fold_no}/{n_splits}]")
    X_train_fold, X_val_fold = X_train[train_index], X_train[val_index]
    y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]
    y_train_text_fold, y_val_text_fold = y_train_text[train_index], y_train_text[val_index]


    att  = MultiOutputClassifier(KNeighborsClassifier(n_neighbors=1, n_jobs=-1, leaf_size=100))
    att.fit(X_train_fold, y_train_text_fold)
    # Generating synthetic data using different methods
    real_loader = xy_to_tensordataset(X_val_fold, y_val_fold, return_loader=True)
            
    synthetic_methods = {
        # 'SMOTE': (conditional_smote_sampling(X_val_fold, y_val_fold, generation_size, condition=[0, 1, 2])[0], y_val_text_fold),
        # 'KDE': (conditional_kde_sampling(X_val_fold, y_val_fold, generation_size, condition=[0, 1, 2], n_components=min(X_train_fold.shape[1] - 1, generation_size - 1))[0], y_val_text_fold),
        # 'GMM': (conditional_gmm_sampling(X_val_fold, y_val_fold, generation_size, condition=[0, 1, 2])[0], y_val_text_fold),
        'GAN':  (GANs(
                batch_size,
                X_val_fold,
                y_val_fold,
                train_y_full,  # using full training labels as in your original code
                latent_dim,
                condition_dim,
                device,
                gan_epochs
            ).generate(real_loader, generation_size)[0], y_val_text_fold)
    }

    # Evaluating each synthetic data method
    for method_name, (synthetic_x, synthetic_y) in synthetic_methods.items():
        pred = att.predict(synthetic_x)
        # Set device explicitly if provided
        model = model.to(torch.device('cuda'))
        seq_similarity_score, unaligned, hit_rate = compute_seq_similarity(model, synthetic_y, pred, tokenizer)


        break

    break
    fold_no += 1




In [ ]:
unaligned, seq_similarity_score

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
parent_dir = os.path.join(os.getcwd(), '..')
if parent_dir not in sys.path: sys.path.append(parent_dir)
from utility.data import get_loader, EmbeddingDataset
from utility.visuals import *
from model.data_augmentation_tech import conditional_smote_sampling
from model.gan import GANs
from model.knn import KNN
import torch
from torch.utils.data import Subset
from datasets import load_from_disk

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
latent_dim = 128          
condition_dim = 10        
gan_epochs = 20            
generation_sizes = [1000, 10000]  # Multiple generation sizes
batch_size = 32                

# Load dataset and split into train and test
full_dataset = load_from_disk('../data/full_dataset_new', keep_in_memory=True)
split_datasets = full_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_datasets['train']
test_dataset = split_datasets['test']

test_x = np.array(test_dataset['embedding'])
test_y = np.array(test_dataset['labels'])

train_x_full = np.array(train_dataset['embedding'])
train_y_full = np.array(train_dataset['labels'])
train_ds = EmbeddingDataset(train_dataset)
test_ds = EmbeddingDataset(test_dataset)
test_loader = get_loader(test_ds, batch_size=batch_size, shuffle=False)  

sample_sizes = [20, 50, 100, 200, 1000, len(train_ds)]

# Initialize dictionaries to store the results
knn_accuracy_before = {}
knn_accuracy_after = []
accuracy_after_title = []


# Iterate through generation sizes
for generation_size in generation_sizes:
    knn_accuracy_after_gen = {}
    print(f"\nTraining with Generation Size: {generation_size}")
    
    for size in sample_sizes:
        print(f"\n[Real Data Only] Training size: {size}")
        X_train = train_x_full[:size]
        y_train = train_y_full[:size]

        # KNN on real data
        print(y_train)
        new_x, new_y = conditional_smote_sampling(X_train, y_train, size)
        print(new_y.shape, new_x.shape)
        raise Exception
        knn_accuracy_before[size] = \
            KNN().fit_and_eval(X_train, y_train, test_x, test_y)

        # Prepare GAN data loader
        train_subset = Subset(train_ds, range(size))
        gan_loader = get_loader(train_subset, batch_size=batch_size, shuffle=True)
        input_dim = X_train.shape[1]  
        unique_labels = np.unique(train_y_full)
        num_classes_gan = len(unique_labels)
        synthetic_x, synthetic_y  = conditional_smote_sampling(

        )
        synthetic_x, synthetic_y = GANs(
            train_ds,
            batch_size,
            X_train,
            y_train,
            train_y_full,
            latent_dim,
            condition_dim,
            device,
            gan_epochs
        ).generate(size, generation_size)
        train_combined_x = np.concatenate([X_train, synthetic_x], axis=0)
        train_combined_y = np.concatenate([y_train, synthetic_y], axis=0)

        # KNN on augmented data (real + synthetic)
        knn_accuracy_after_gen[size] = \
            KNN().fit_and_eval(
                train_combined_x, train_combined_y, test_x, test_y
            )
    knn_accuracy_after.append(knn_accuracy_after_gen)
    accuracy_after_title.append(f"After Concatenation Accuracy with size of {generation_size}")

# Display the accuracy summary
# display_accuracy_summary_with_more_than_two_bars(summary_df, file_name="../../figure/generation_sizes_accuracy_plot.png")
# save_table_as_image(summary_df, file_name="../../figure/generation_sizes_accuracy.png")

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer, BertModel

# ?????????
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = pd.read_csv("SICK_filtered.tsv", sep="\t")
print("??????:", df.shape)

# ? entailment_label ? stratified ??:80% ??,20% ??
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["entailment_label"])
print("?????:", df_train.shape, "?????:", df_test.shape)

# ????:? entailment_label ?????
le = LabelEncoder()
df_train["label_enc"] = le.fit_transform(df_train["entailment_label"])
df_test["label_enc"] = le.transform(df_test["entailment_label"])
num_classes = len(le.classes_)
print("????:", le.classes_)

pretrained_model = "bert-base-uncased"  # ??????,??? 'bert-base-chinese'
tokenizer = BertTokenizer.from_pretrained(pretrained_model)

# ????????
max_length = 128


class SICKBERTDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.sentences_A = df["sentence_A"].tolist()
        self.sentences_B = df["sentence_B"].tolist()
        self.labels = df["label_enc"].tolist()  # ??????????
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        text_A = self.sentences_A[idx]
        text_B = self.sentences_B[idx]
        label = self.labels[idx]
        # ? sentence_A ?? tokenize
        encoding_A = self.tokenizer(
            text_A,
            add_special_tokens=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        # ? sentence_B ?? tokenize
        encoding_B = self.tokenizer(
            text_B,
            add_special_tokens=True,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        # ?? batch ?? (??????)
        input_ids_A = encoding_A["input_ids"].squeeze(0)
        attention_mask_A = encoding_A["attention_mask"].squeeze(0)
        input_ids_B = encoding_B["input_ids"].squeeze(0)
        attention_mask_B = encoding_B["attention_mask"].squeeze(0)
        
        return {
            "input_ids_A": input_ids_A,
            "attention_mask_A": attention_mask_A,
            "input_ids_B": input_ids_B,
            "attention_mask_B": attention_mask_B,
            "label": torch.tensor(label, dtype=torch.long)
        }

def create_dataloader(df, tokenizer, max_length, batch_size, shuffle=True):
    dataset = SICKBERTDataset(df, tokenizer, max_length)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


class BertDNNClassifier(nn.Module):
    def __init__(self, pretrained_model_name, hidden_dim, num_classes, dropout=0.5):
        super(BertDNNClassifier, self).__init__()
        # ??? sentence_A ? sentence_B ????? BERT encoder
        self.bert_A = BertModel.from_pretrained(pretrained_model_name)
        self.bert_B = BertModel.from_pretrained(pretrained_model_name)
        self.dropout = nn.Dropout(dropout)
        # ? BERT ? pooler_output (768 ?) ???? 1536 ?
        self.fc = nn.Sequential(
            nn.Linear(768 * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
    
    def forward(self, input_ids_A, attention_mask_A, input_ids_B, attention_mask_B):
        outputs_A = self.bert_A(input_ids=input_ids_A, attention_mask=attention_mask_A)
        pooled_A = outputs_A.pooler_output  # (batch, 768)
        
        outputs_B = self.bert_B(input_ids=input_ids_B, attention_mask=attention_mask_B)
        pooled_B = outputs_B.pooler_output  # (batch, 768)
        
        # ??????(???)
        features = torch.cat([pooled_A, pooled_B], dim=1)  # (batch, 1536)
        logits = self.fc(features)
        return logits


def train_and_evaluate(model, train_loader, test_loader, num_epochs, lr):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    train_losses, test_losses = [], []
    train_accuracies, test_accuracies = [], []
    epoch_list = []
    
    for epoch in range(1, num_epochs + 1):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for batch in train_loader:
            input_ids_A = batch["input_ids_A"].to(device)
            attention_mask_A = batch["attention_mask_A"].to(device)
            input_ids_B = batch["input_ids_B"].to(device)
            attention_mask_B = batch["attention_mask_B"].to(device)
            labels = batch["label"].to(device)
            
            optimizer.zero_grad()
            outputs = model(input_ids_A, attention_mask_A, input_ids_B, attention_mask_B)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * labels.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
        
        train_loss = running_loss / total
        train_acc = correct / total
        
        model.eval()
        test_running_loss = 0.0
        test_correct = 0
        test_total = 0
        with torch.no_grad():
            for batch in test_loader:
                input_ids_A = batch["input_ids_A"].to(device)
                attention_mask_A = batch["attention_mask_A"].to(device)
                input_ids_B = batch["input_ids_B"].to(device)
                attention_mask_B = batch["attention_mask_B"].to(device)
                labels = batch["label"].to(device)
                
                outputs = model(input_ids_A, attention_mask_A, input_ids_B, attention_mask_B)
                loss = criterion(outputs, labels)
                test_running_loss += loss.item() * labels.size(0)
                _, preds = torch.max(outputs, 1)
                test_correct += (preds == labels).sum().item()
                test_total += labels.size(0)
        
        test_loss = test_running_loss / test_total
        test_acc = test_correct / test_total
        
        epoch_list.append(epoch)
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        train_accuracies.append(train_acc)
        test_accuracies.append(test_acc)
        
        print(f"Epoch {epoch}: Train Loss={train_loss:.4f}, Train Acc={train_acc:.4f} | Test Loss={test_loss:.4f}, Test Acc={test_acc:.4f}")
    
    return epoch_list, train_losses, test_losses, train_accuracies, test_accuracies


batch_size = 16       
num_epochs = 5         
learning_rate = 2e-5   

X = df_train.copy()

fractions = [1.0, 0.5, 0.25, 0.1]

for frac in fractions:
    num_samples = int(len(X) * frac)
    df_train_subset = X.iloc[:num_samples].reset_index(drop=True)
    subset_percentage = int(frac * 100)
    df_train_subset.to_csv(f"SICK_train_subset_{subset_percentage}.tsv", sep="\t", index=False)
    
    train_loader = create_dataloader(df_train_subset, tokenizer, max_length, batch_size, shuffle=True)
    test_loader = create_dataloader(df_test, tokenizer, max_length, batch_size, shuffle=False)
    
    print(f"\n[?? {subset_percentage}%] ??????: {len(df_train_subset)}")
    
    # ??? BERT+DNN ??
    model = BertDNNClassifier(pretrained_model, hidden_dim=256, num_classes=num_classes, dropout=0.5)
    model.to(device)
    
    epoch_list, train_losses, test_losses, train_accs, test_accs = train_and_evaluate(model, train_loader, test_loader, num_epochs, learning_rate)
    
    # ??:Loss ? Accuracy ? Epoch ??
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    axs[0].plot(epoch_list, train_losses, label="Train Loss")
    axs[0].plot(epoch_list, test_losses, label="Test Loss")
    axs[0].set_xlabel("Epoch")
    axs[0].set_ylabel("Loss")
    axs[0].set_title(f"Loss vs Epoch (Train Subset {subset_percentage}%)")
    axs[0].legend()
    
    axs[1].plot(epoch_list, train_accs, label="Train Acc")
    axs[1].plot(epoch_list, test_accs, label="Test Acc")
    axs[1].set_xlabel("Epoch")
    axs[1].set_ylabel("Accuracy")
    axs[1].set_title(f"Accuracy vs Epoch (Train Subset {subset_percentage}%)")
    axs[1].legend()
    
    plt.tight_layout()
    plt.savefig(f"BertDNN_SICK_{subset_percentage}.png")
    plt.close()
    
    # ???????????
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for batch in test_loader:
            input_ids_A = batch["input_ids_A"].to(device)
            attention_mask_A = batch["attention_mask_A"].to(device)
            input_ids_B = batch["input_ids_B"].to(device)
            attention_mask_B = batch["attention_mask_B"].to(device)
            labels = batch["label"].to(device)
            
            outputs = model(input_ids_A, attention_mask_A, input_ids_B, attention_mask_B)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    print(f"???? (Train Subset {subset_percentage}%):")
    print(classification_report(all_labels, all_preds, target_names=le.classes_))
    from sklearn.metrics import confusion_matrix

    cm = confusion_matrix(all_labels, all_preds)
    print("Confusion Matrix:")
    print(cm)

    plt.figure(figsize=(6, 6))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title("Confusion Matrix")
    plt.colorbar()
    tick_marks = np.arange(len(le.classes_))
    plt.xticks(tick_marks, le.classes_, rotation=45)
    plt.yticks(tick_marks, le.classes_)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()
